In [2]:
import os

dataset_path = "datasets"

for cls in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, cls)

    if os.path.isdir(class_path):
        count = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

        print(f"{cls}: {count} images")

abnormal: 5988 images
normal: 6104 images


In [3]:
import os
import cv2

input_dir = "datasets"
output_dir = "resized_dataset"

IMG_SIZE = (224, 224)

os.makedirs(output_dir, exist_ok=True)

for class_name in os.listdir(input_dir):

    class_input = os.path.join(input_dir, class_name)

    if not os.path.isdir(class_input):
        continue

    class_output = os.path.join(output_dir, class_name)
    os.makedirs(class_output, exist_ok=True)

    for image_name in os.listdir(class_input):

        image_path = os.path.join(class_input, image_name)

        img = cv2.imread(image_path)

        if img is None:
            print(f"Skipping: {image_path}")
            continue

        # Resize image
        resized_img = cv2.resize(
            img,
            IMG_SIZE,
            interpolation=cv2.INTER_AREA
        )

        save_path = os.path.join(class_output, image_name)

        cv2.imwrite(save_path, resized_img)

print("All images resized successfully!")

All images resized successfully!


## Split the Dataset

In [5]:
import os
import shutil
from sklearn.model_selection import train_test_split

source_dir = "resized_dataset"
output_dir = "skin_dataset"

classes = ["normal", "abnormal"]

for cls in classes:

    class_dir = os.path.join(source_dir, cls)

    images = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    # 70% train, 30% temp
    train_files, temp_files = train_test_split(
        images,
        test_size=0.30,
        random_state=42
    )

    # Split temp into 15% val + 15% test
    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=42
    )

    for split_name, file_list in [
        ("train", train_files),
        ("val", val_files),
        ("test", test_files)
    ]:

        split_dir = os.path.join(output_dir, split_name, cls)
        os.makedirs(split_dir, exist_ok=True)

        for file in file_list:
            src = os.path.join(class_dir, file)
            dst = os.path.join(split_dir, file)
            shutil.copy2(src, dst)

print("Dataset split completed.")

Dataset split completed.


## Load the Model

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "skin_dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "skin_dataset/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "skin_dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 8463 files belonging to 2 classes.
Found 1814 files belonging to 2 classes.
Found 1815 files belonging to 2 classes.


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## Build CNN

In [10]:
from tensorflow.keras import layers, models

model = models.Sequential([

    layers.Rescaling(1./255, input_shape=(224,224,3)),

    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 109, 109, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 52, 52, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     2,359,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,456,993 (9.37 MB)

 Trainable params: 2,456,993 (9.37 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_skin_model.keras",
    monitor='val_accuracy',
    save_best_only=True
)

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
    # callbacks=[early_stop, checkpoint]
)

Epoch 1/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 53s 191ms/step - accuracy: 0.9296 - loss: 0.1765 - val_accuracy: 0.9730 - val_loss: 0.0837
Epoch 2/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 38s 142ms/step - accuracy: 0.9799 - loss: 0.0578 - val_accuracy: 0.9851 - val_loss: 0.0450
Epoch 3/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 39s 148ms/step - accuracy: 0.9897 - loss: 0.0317 - val_accuracy: 0.9884 - val_loss: 0.0356
Epoch 4/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 41s 155ms/step - accuracy: 0.9928 - loss: 0.0209 - val_accuracy: 0.9912 - val_loss: 0.0266
Epoch 5/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 41s 155ms/step - accuracy: 0.9946 - loss: 0.0180 - val_accuracy: 0.9901 - val_loss: 0.0361
Epoch 6/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 43s 161ms/step - accuracy: 0.9918 - loss: 0.0251 - val_accuracy: 0.9873 - val_loss: 0.0366
Epoch 7/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 43s 163ms/step - accuracy: 0.9967 - loss: 0.0090 - val_accuracy: 0.9950 - val_loss: 0.0251
Epoch 8/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 45s 168ms/step - accuracy: 0.9985 - loss: 0

KeyboardInterrupt: 